<a href="https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzFit_User_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THzFit User Workflow

Use this notebook to fit measured THz reference/sample traces with the THzSim2 time-domain fitter.

This version is designed for Google Colab and non-programmer use:
- common settings are edited with simple form fields
- arbitrary multilayer samples are defined with one validated JSON block
- the notebook exports a `fit_setup.json` file that you can send back for review
- the preprocessing preview shows the raw traces, aligned traces, and processed traces used by the fitter


## 0. Install And Import

Run the next cell first. In Colab it installs the package from GitHub automatically. Locally it also works when the notebook is opened inside the repository.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PACKAGE_IMPORT_OK = False
IMPORT_EXCEPTION = None
DEFAULT_PIP_SPEC = 'https://github.com/Podrimate/THz_sim_application/archive/refs/heads/main.zip'

def try_import_thzsim2():
    global IMPORT_EXCEPTION
    try:
        importlib.invalidate_caches()
        import thzsim2  # noqa: F401
        return thzsim2
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return None

if IN_COLAB:
    pip_spec = os.environ.get('THZSIM_PIP_SPEC', DEFAULT_PIP_SPEC).strip()
    subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'thzsim2'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-cache-dir', pip_spec])
    sys.modules.pop('thzsim2', None)
    sys.modules.pop('thzsim2.notebook_api', None)

thzsim2 = try_import_thzsim2()
PACKAGE_IMPORT_OK = thzsim2 is not None

if not PACKAGE_IMPORT_OK:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'thzsim2').exists():
            sys.path.insert(0, str(candidate))
            thzsim2 = try_import_thzsim2()
            if thzsim2 is not None:
                PACKAGE_IMPORT_OK = True
                break

if not PACKAGE_IMPORT_OK:
    raise RuntimeError(
        'Could not import thzsim2. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab restart the runtime and rerun the first cell; locally open the notebook inside the repo.'
    )

print(f'Using thzsim2 from: {thzsim2.__file__}')
print(f'thzsim2 version: {getattr(thzsim2, "__version__", "unknown")}')


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from thzsim2.io.trace_csv import read_trace_csv
from thzsim2.notebook_api import (
    build_fit_setup,
    load_fit_setup_json,
    plot_trace_pair_preview,
    run_measured_fit_from_setup_json,
    summarize_prepared_trace_pair,
    summarize_trace_input,
    write_fit_setup_json,
)

workflow_root = Path.cwd() / 'thz_fit_workflow_outputs'
uploads_root = workflow_root / 'uploads'
workflow_root.mkdir(parents=True, exist_ok=True)
uploads_root.mkdir(parents=True, exist_ok=True)

def find_repo_path(relative_path):
    for candidate in [Path.cwd(), Path.cwd().parent]:
        candidate_path = candidate / relative_path
        if candidate_path.exists():
            return candidate_path.resolve()
    return None

LOCAL_TEST_DATA = find_repo_path('Test_data_for_fitter')

def upload_single_csv(target_dir, *, prompt):
    print(prompt)
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError('Colab upload is only available inside Google Colab.') from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    name, payload = next(iter(uploaded.items()))
    path = Path(target_dir) / name
    path.write_bytes(payload)
    return path.resolve()

def parse_json_block(text, *, label):
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(f'{label} JSON is invalid at line {exc.lineno}, column {exc.colno}: {exc.msg}') from exc

def show_mapping(title, mapping):
    display(Markdown(f'**{title}**'))
    for key, value in mapping.items():
        print(f'- {key}: {value}')

def resolve_example_pair(dataset_name, sample_name):
    if LOCAL_TEST_DATA is None:
        return None, None
    reference_csv = LOCAL_TEST_DATA / dataset_name / 'REFERENCE.csv'
    sample_csv = LOCAL_TEST_DATA / dataset_name / sample_name
    if not reference_csv.exists() or not sample_csv.exists():
        return None, None
    return reference_csv.resolve(), sample_csv.resolve()


## 1. Choose The Reference And Sample CSV Files

What these parameters mean:
- `use_colab_upload`: use file-pick dialogs in Colab. This is the easiest path for most users.
- `use_local_example_if_available`: outside Colab, try the repository example data automatically.
- `reference_csv_path` and `sample_csv_path`: manual file paths if you already know them.

What to change:
- In Colab: set `use_colab_upload=True` and leave the paths blank.
- Locally: either use the example data or paste the full CSV paths.


In [ ]:
use_colab_upload = False #@param {type:"boolean"}
use_local_example_if_available = True #@param {type:"boolean"}
example_dataset = "A11008858_transmission" #@param ["A11008858_transmission", "A11013460_transmission"]
example_sample_name = "SAMPLE.csv" #@param ["SAMPLE.csv", "SAMPLE1.csv", "SAMPLE2.csv"]
reference_csv_path = "" #@param {type:"string"}
sample_csv_path = "" #@param {type:"string"}


In [ ]:
selected_reference_csv = Path(reference_csv_path).expanduser().resolve() if reference_csv_path.strip() else None
selected_sample_csv = Path(sample_csv_path).expanduser().resolve() if sample_csv_path.strip() else None

if use_local_example_if_available and (selected_reference_csv is None or selected_sample_csv is None):
    example_reference_csv, example_sample_csv = resolve_example_pair(example_dataset, example_sample_name)
    if selected_reference_csv is None:
        selected_reference_csv = example_reference_csv
    if selected_sample_csv is None:
        selected_sample_csv = example_sample_csv

if use_colab_upload:
    selected_reference_csv = upload_single_csv(uploads_root, prompt='Upload the measured reference CSV file.')
    selected_sample_csv = upload_single_csv(uploads_root, prompt='Upload the measured sample CSV file.')

if selected_reference_csv is None or selected_sample_csv is None:
    raise RuntimeError('No reference/sample files are selected. Use Colab upload, the local example, or paste file paths.')

reference_import_summary = summarize_trace_input(selected_reference_csv)
sample_import_summary = summarize_trace_input(selected_sample_csv)
show_mapping('Reference Import Summary', reference_import_summary)
show_mapping('Sample Import Summary', sample_import_summary)

def quick_plot_from_csv(path, label):
    trace = read_trace_csv(path, resample='auto')
    plt.plot(trace.time_ps, trace.trace, label=label)

plt.figure(figsize=(10, 4))
quick_plot_from_csv(selected_reference_csv, 'raw reference')
quick_plot_from_csv(selected_sample_csv, 'raw sample')
plt.title('Raw Uploaded Traces')
plt.xlabel('Time')
plt.ylabel('Signal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 2. Preprocessing Settings

What these parameters mean:
- `baseline_mode='auto_pre_pulse'`: estimate the baseline from quiet data before the earliest strong pulse. This is the safe default.
- `baseline_mode='first_samples'`: use the first `baseline_window_samples` values as the baseline region.
- `crop_mode='auto'`: keep a window that contains both dominant pulses plus extra pre/post pulse margin. This is the recommended default.
- `crop_mode='manual'`: use your exact crop window. Only use this if you are sure the main pulse stays inside the window.

Common failure symptom:
- if the processed trace loses the biggest pulse, the fit will often chase ringing or noise instead of the real signal.


In [ ]:
baseline_mode = "auto_pre_pulse" #@param ["auto_pre_pulse", "first_samples", "none"]
baseline_window_samples = 40 #@param {type:"integer"}
crop_mode = "auto" #@param ["auto", "manual", "none"]
manual_crop_min_ps = 2605.0 #@param {type:"number"}
manual_crop_max_ps = 2645.0 #@param {type:"number"}


In [ ]:
from thzsim2.notebook_api import prepare_trace_pair_for_fit

resolved_crop_window_ps = (manual_crop_min_ps, manual_crop_max_ps) if crop_mode == 'manual' else None
prepared_traces = prepare_trace_pair_for_fit(
    selected_reference_csv,
    selected_sample_csv,
    baseline_subtract=(baseline_mode != 'none'),
    baseline_window_samples=baseline_window_samples,
    crop_time_window_ps=resolved_crop_window_ps,
    baseline_mode=baseline_mode,
    crop_mode=crop_mode,
)
prepared_summary = summarize_prepared_trace_pair(prepared_traces)
show_mapping('Prepared Trace Summary', prepared_summary)
if prepared_summary['warnings']:
    display(Markdown('**Warning:** ' + ' '.join(prepared_summary['warnings'])))
plot_trace_pair_preview(prepared_traces)


## 3. Measurement Model

What these parameters mean:
- `measurement_mode`: choose `transmission` or `reflection`.
- `angle_mode`: keep the incident angle fixed or fit it.
- `polarization_model='mixed'`: use one effective s/p mixing parameter when the source and detector polarization states are not fully known.
- `reference_standard_kind='identity'`: the uploaded reference is treated as the physical reference. This is the usual setting for the measured-fit workflow.
- `reference_standard_kind='stack'`: divide by a modeled reference standard stack before comparing to the sample.


In [ ]:
measurement_mode = "transmission" #@param ["transmission", "reflection"]
angle_mode = "fit" #@param ["fixed", "fit"]
angle_initial_deg = 8.0 #@param {type:"number"}
angle_min_deg = 0.0 #@param {type:"number"}
angle_max_deg = 25.0 #@param {type:"number"}
polarization_model = "mixed" #@param ["s", "p", "mixed"]
mix_mode = "fit" #@param ["fixed", "fit"]
mix_initial = 0.5 #@param {type:"number"}
mix_min = 0.0 #@param {type:"number"}
mix_max = 1.0 #@param {type:"number"}
reference_standard_kind = "identity" #@param ["identity", "stack"]


## 4. Sample Definition JSON

Edit the JSON in the next cell instead of writing Python code.

Important schema rules:
- `layers` is a list, from incident side to exit side
- `thickness_um` can be a number or a `Fit` block
- material parameters can also be numbers or `Fit` blocks
- a `Fit` block looks like:
  `{"kind": "Fit", "initial": 550.0, "abs_min": 300.0, "abs_max": 800.0, "label": "film_thickness_um"}`

Safe starting point:
- keep the example structure first, verify the preprocessing plot, then change one material/parameter group at a time.


In [ ]:
sample_n_in = 1.0 #@param {type:"number"}
sample_n_out = 1.0 #@param {type:"number"}
overlay_imported_nk = True #@param {type:"boolean"}

sample_layers_json = r'''
[
  {
    "name": "drude_layer",
    "thickness_um": {
      "kind": "Fit",
      "initial": 550.0,
      "abs_min": 300.0,
      "abs_max": 800.0,
      "label": "film_thickness_um"
    },
    "material": {
      "kind": "Drude",
      "eps_inf": 12.0,
      "plasma_freq_thz": {
        "kind": "Fit",
        "initial": 1.1,
        "abs_min": 0.1,
        "abs_max": 3.0,
        "label": "film_plasma_freq_thz"
      },
      "gamma_thz": {
        "kind": "Fit",
        "initial": 0.08,
        "abs_min": 0.005,
        "abs_max": 0.5,
        "label": "film_gamma_thz"
      }
    }
  }
]
'''

reference_standard_stack_json = r'''
{
  "layers": []
}
'''


## 5. Fit Controls And Export Files

What these parameters mean:
- `metric`: the objective function minimized during fitting
- `max_internal_reflections`: include more internal reflections for thicker or more resonant stacks, but expect slower runs
- `fit_setup_path`: the JSON file you can save and send back for review
- `fit_output_dir`: where the fit outputs are written


In [ ]:
metric = "mse" #@param ["mse", "relative_l2"]
max_internal_reflections = 2 #@param {type:"integer"}
optimizer_maxiter = 8 #@param {type:"integer"}
optimizer_global_maxiter = 1 #@param {type:"integer"}
optimizer_popsize = 5 #@param {type:"integer"}
optimizer_seed = 11 #@param {type:"integer"}
fit_setup_path = str((workflow_root / 'fit_setup.json').resolve()) #@param {type:"string"}
fit_output_dir = str((workflow_root / 'fit_run').resolve()) #@param {type:"string"}


In [ ]:
sample_layers = parse_json_block(sample_layers_json, label='Sample layers')
reference_standard_stack = parse_json_block(reference_standard_stack_json, label='Reference standard stack')

angle_value = angle_initial_deg if angle_mode == 'fixed' else {
    'kind': 'Fit',
    'initial': angle_initial_deg,
    'abs_min': angle_min_deg,
    'abs_max': angle_max_deg,
    'label': 'measurement_angle_deg',
}
if polarization_model == 'mixed':
    mix_value = mix_initial if mix_mode == 'fixed' else {
        'kind': 'Fit',
        'initial': mix_initial,
        'abs_min': mix_min,
        'abs_max': mix_max,
        'label': 'measurement_polarization_mix',
    }
else:
    mix_value = None

measurement_config = {
    'mode': measurement_mode,
    'angle_deg': angle_value,
    'polarization': polarization_model,
    'polarization_mix': mix_value,
    'reference_standard': {'kind': 'identity'} if reference_standard_kind == 'identity' else {'kind': 'stack', 'stack': reference_standard_stack},
}

fit_setup = build_fit_setup(
    reference_trace={'path': selected_reference_csv},
    sample_trace={'path': selected_sample_csv},
    layers=sample_layers,
    preprocessing={
        'baseline_subtract': baseline_mode != 'none',
        'baseline_mode': baseline_mode,
        'baseline_window_samples': baseline_window_samples,
        'crop_mode': crop_mode,
        'crop_time_window_ps': resolved_crop_window_ps,
    },
    measurement=measurement_config,
    optimizer={
        'method': 'L-BFGS-B',
        'options': {'maxiter': int(optimizer_maxiter)},
        'global_options': {'maxiter': int(optimizer_global_maxiter), 'popsize': int(optimizer_popsize), 'seed': int(optimizer_seed)},
        'fd_rel_step': 1e-4,
    },
    metric=metric,
    max_internal_reflections=max_internal_reflections,
    n_in=sample_n_in,
    n_out=sample_n_out,
    overlay_imported=overlay_imported_nk,
    out_dir=fit_output_dir,
)
fit_setup_json_path = write_fit_setup_json(fit_setup_path, fit_setup)
show_mapping('Saved Fit Setup', {'fit_setup_json': fit_setup_json_path})
display(Markdown('You can send `fit_setup.json` back for review.'))


## 6. Run The Fit And Review The Result

What you should see after running the next two cells:
- the processed sample and the fit should overlap well in the main pulse region
- the residual should be much smaller than the fitted pulse amplitude
- the optical response should look physically smooth, not wildly noisy from one frequency sample to the next


In [ ]:
measured_fit_result = run_measured_fit_from_setup_json(fit_setup_json_path)
fit_summary = {
    'objective_value': float(measured_fit_result.fit_result['objective_value']),
    'metric': measured_fit_result.fit_result['metric'],
    'output_dir': str(measured_fit_result.out_dir),
    'setup_json': str(fit_setup_json_path),
}
show_mapping('Fit Summary', fit_summary)
show_mapping('Recovered Parameters', measured_fit_result.fit_result['recovered_parameters'])


In [ ]:
fit_trace = np.asarray(measured_fit_result.fit_result['fitted_simulation']['sample_trace'], dtype=float)
processed_sample = measured_fit_result.prepared_traces.processed_sample
processed_reference = measured_fit_result.prepared_traces.processed_reference
residual = np.asarray(measured_fit_result.fit_result['residual_trace'], dtype=float)
omega = np.asarray(measured_fit_result.fit_result['fitted_simulation']['omega_rad_s'], dtype=float)
freq_thz = omega / (2.0 * np.pi * 1e12)
transfer = np.asarray(measured_fit_result.fit_result['fitted_simulation']['transfer_function'])

fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=False)
axes[0].plot(processed_reference.time_ps, processed_reference.trace, label='processed reference', alpha=0.8)
axes[0].plot(processed_sample.time_ps, processed_sample.trace, label='processed sample', linewidth=1.4)
axes[0].plot(processed_sample.time_ps, fit_trace, label='fit', linewidth=1.4)
axes[0].set_title('Measured Time-Domain Fit')
axes[0].set_xlabel('Time (ps)')
axes[0].set_ylabel('Signal')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(processed_sample.time_ps, residual, label='residual')
axes[1].set_title('Residual Trace')
axes[1].set_xlabel('Time (ps)')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(freq_thz, np.abs(transfer), label='|sample response|')
axes[2].set_title('Optical Response Magnitude')
axes[2].set_xlabel('Frequency (THz)')
axes[2].set_ylabel('Magnitude')
axes[2].grid(True, alpha=0.3)
axes[2].legend()
fig.tight_layout()
plt.show()


## 7. Files To Keep Or Send Back

The most important review file is the exported `fit_setup.json`.

If you want me to inspect the run later, send:
- `fit_setup.json`
- optionally the exported processed/fitted CSV files from the fit output directory
- optionally screenshots of the preprocessing preview and fit plots
